In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt

from shapely.geometry import box

from skimage.registration import phase_cross_correlation

In [6]:
series_folder = Path(r".\data\test-series")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_462189531\ROI-w1-20nm-bsd")
series_folder = Path(r"Z:\zeinab\ATLAS-projects\atlas_L32-10-1\atlas_L32-10-1_data\session_2053739366\ROI-w1-20nm-bsd")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_955928929\w3-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_L32-10-2\atlas_L32-10-2_20250410_data\session_1395343682\w2-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_862126381\Section Set 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\TA31-luke_data\session_402716765\ta31-roi1-2")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA29\Site 3")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA30\ROI")
series_folder = Path(r'E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_olive_2\zeinab-olive-2-main_data\session_598049588\roi-01')
series_folder = Path(r'E:\PROJECTS\EM\Filipa\M2-2\whole sample sections 20251112_data\session_1077734160\Site 2')

series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\Filipa-M2-2-20251127\sections 8-12_data\session_1422202770\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant-3-wafer-1\alma-plant-3_data\session_422002753\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant2-wafer-1\plant 2_data\session_1115605826\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant4-wafer-1\alma-plant-4_data\session_1084385689\Site 1")

series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\test\Site 2")

series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\ATLAS-projects\proj-4-20260302\NEW LOCATION20260303_data\session_1149164643\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2")


series_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        tif_files = list(folder.glob("*.tif"))
        if tif_files:  # Check if the list is not empty
            print(f"Found series folder: {folder.name} (contains {len(tif_files)} .tif files)")
            series_list.append(folder)

Found series folder: S_001_1502294429 (contains 15 .tif files)
Found series folder: S_002_104968728 (contains 10 .tif files)
Found series folder: S_003_1474649724 (contains 10 .tif files)
Found series folder: S_004_1545571955 (contains 10 .tif files)
Found series folder: S_005_632395844 (contains 10 .tif files)
Found series folder: S_006_891243350 (contains 10 .tif files)
Found series folder: S_007_1059935411 (contains 10 .tif files)
Found series folder: S_008_1563112833 (contains 10 .tif files)
Found series folder: S_009_1434391433 (contains 10 .tif files)
Found series folder: S_010_568511412 (contains 10 .tif files)
Found series folder: S_011_538922633 (contains 10 .tif files)
Found series folder: S_012_90664623 (contains 10 .tif files)
Found series folder: S_013_1704454511 (contains 10 .tif files)
Found series folder: S_014_534352178 (contains 10 .tif files)
Found series folder: S_015_1874207230 (contains 10 .tif files)
Found series folder: S_016_605186045 (contains 10 .tif files)
F

In [3]:
from collections import defaultdict, deque
from atlas.stitching import get_tiles_dataframe, stitch_ATLAS_tiles
from atlas.stitching import add_tile_overlap_columns, match_tiles, build_adjacency_matrix_from_costs, build_transform_dict_from_mst, apply_transforms_and_stitch


In [4]:
from atlas.stitching import get_overlap_relative, mask_low_and_saturation
def match_tiles(input_df, reference_idx, min_overlap_percent=2, std_th=2, do_hanning=True):
    """
    Compute stitching cost and pixel-shift vectors between a reference tile and all other tiles.

    For each tile, the function:
      1) checks the precomputed overlap percentage against `min_overlap_percent`,
      2) extracts the overlapping image regions using the tiles' geometries,
      3) builds masks to exclude low-value and saturated pixels,
      4) optionally refines the masks by keeping only pixels above a global
         intensity threshold derived from mean + std_th * std of valid pixels,
      5) estimates translation via masked phase cross-correlation,
      6) computes a heuristic stitching cost.

    Parameters
    ----------
    input_df : pandas.DataFrame
        DataFrame containing tile metadata. Must include:
        - 'geometry' : shapely geometry (tile bounding box in global space)
        - 'ImageWidth' : width in pixels
        - 'ImageHeight' : height in pixels
        - 'Filename' : TIFF filename
        - 'raw_data_folder' : pathlib.Path to folder containing the raw TIFF
        - 'overlap_percent' : list-like of overlap percentages with all other tiles
    reference_idx : int
        Index of the tile used as the reference for cost and shift computation.
    min_overlap_percent : float, optional (default=2)
        Minimum overlap (%) required to attempt matching. Tiles below this threshold
        get cost=1.0 and shift=[0, 0].
    std_th : float or None or False, optional (default=2)
        Optional intensity-based mask refinement.
        - If None or False: do not apply intensity thresholding.
        - If a number: compute pix_th = mean + std_th * std using valid pixels
          from both overlap crops and keep only pixels > pix_th in both masks.
        Note: This assumes foreground is brighter than background; for inverted
        contrast (e.g., some BSD images), this may reject signal.

    Returns
    -------
    cost_list : list of float
        One stitching cost per tile.
    shift_list : list of np.ndarray, shape (2,)
        Detected pixel shifts (row, col) aligning each tile to the reference tile.
        If overlap is too small, the shift is np.zeros(2).

    Notes
    -----
    - Shifts follow the `skimage.registration.phase_cross_correlation` convention
      (row, col). Apply with care regarding sign depending on your downstream usage.
    - Masked phase cross-correlation may return NaNs for the error metric (known behavior);
      this function does not use that error value.
    """

    # ------------------------------------------------------------------
    # Assertions: Validate DataFrame structure
    # ------------------------------------------------------------------
    required_cols = [
        'geometry', 'ImageWidth', 'ImageHeight',
        'Filename', 'raw_data_folder', 'overlap_percent'
    ]
    for col in required_cols:
        assert col in input_df.columns, f"Missing required column: '{col}'"

    assert 0 <= reference_idx < len(input_df), "reference_idx is out of DataFrame bounds"
    assert isinstance(min_overlap_percent, (int, float)), "min_overlap_percent must be numeric"

    # ------------------------------------------------------------------
    # Setup reference tile
    # ------------------------------------------------------------------
    row_ref = input_df.iloc[reference_idx]
    geometry_ref = row_ref['geometry']
    w_ref = row_ref['ImageWidth']
    h_ref = row_ref['ImageHeight']

    # Load dtype from TIFF
    ref_tif_path = row_ref.raw_data_folder.joinpath(Path(row_ref.Filename).name)
    with tiff.TiffFile(ref_tif_path) as tif:
        image_dtype = tif.pages[0].dtype

    print(f"\nProcessing reference tile {reference_idx}...")

    # Prepare outputs
    n_tiles = len(input_df)
    cost_list = []
    shift_list = []

    # ------------------------------------------------------------------
    # Compare reference tile to all other tiles
    # ------------------------------------------------------------------
    for query_idx in range(n_tiles):
        print(f"\nComparing reference {reference_idx} to tile {query_idx}...")

        row_mov = input_df.iloc[query_idx]
        overlap_percentage = row_ref.overlap_percent[query_idx]

        print(f"Overlap %: {overlap_percentage}")

        # Not enough overlap → default cost/shift
        if overlap_percentage < min_overlap_percent:
            print("Too little overlap: assigning cost=1.0 and shift=[0,0]")
            cost_list.append(np.float64(1.0))
            shift_list.append(np.zeros(2))
            continue

        # ------------------------------------------------------------------
        # Compute overlapping bounding boxes
        # ------------------------------------------------------------------
        geometry_mov = row_mov['geometry']
        mov_tif_path = row_ref.raw_data_folder.joinpath(Path(row_mov.Filename).name)

        ref_box, mov_box = get_overlap_relative(
            box_reference=geometry_ref,
            box_moving=geometry_mov
        )

        # ------------------------------------------------------------------
        # Load overlapping region from reference image
        # ------------------------------------------------------------------
        with tiff.TiffFile(ref_tif_path) as tif:
            y0_tmp, y1_tmp = int(ref_box.bounds[1]), int(ref_box.bounds[3])
            y0, y1 = h_ref - y1_tmp, h_ref - y0_tmp  # flip correction
            x0, x1 = int(ref_box.bounds[0]), int(ref_box.bounds[2])
            crop_ref = np.flipud(tif.asarray()[y0:y1, x0:x1])

        # ------------------------------------------------------------------
        # Load overlapping region from moving image
        # ------------------------------------------------------------------
        with tiff.TiffFile(mov_tif_path) as tif:
            y0_tmp, y1_tmp = int(mov_box.bounds[1]), int(mov_box.bounds[3])
            y0, y1 = h_ref - y1_tmp, h_ref - y0_tmp
            x0, x1 = int(mov_box.bounds[0]), int(mov_box.bounds[2])
            crop_mov = np.flipud(tif.asarray()[y0:y1, x0:x1])

        # ------------------------------------------------------------------
        # Mask & threshold computation
        # ------------------------------------------------------------------
        mask_ref = ~mask_low_and_saturation(crop_ref)
        mask_mov = ~mask_low_and_saturation(crop_mov)

        current_vals = np.concatenate([
            crop_ref[mask_ref].ravel(),
            crop_mov[mask_mov].ravel()
        ])

        if current_vals.size == 0:
            # fall back to no intensity thresholding
            pix_mean = pix_std = None
        else:
            pix_mean = current_vals.mean()
            pix_std = current_vals.std()

        if (std_th is not None and std_th is not False) and (current_vals.size > 0):
            pix_th = pix_mean + float(std_th) * pix_std
            mask_mov = np.logical_and(mask_mov, crop_mov > pix_th)
            mask_ref = np.logical_and(mask_ref, crop_ref > pix_th)

        mask_pixels = mask_mov.sum()
        mask_pixels_per = mask_pixels / mask_mov.size

        # optional: reduce edge effects (often helps a lot)
        if do_hanning:
                
            wy = np.hanning(crop_ref.shape[0])
            wx = np.hanning(crop_ref.shape[1])
            win = wy[:, None] * wx[None, :]
            crop_ref = crop_ref * win

            wy = np.hanning(crop_mov.shape[0])
            wx = np.hanning(crop_mov.shape[1])
            win = wy[:, None] * wx[None, :]
            crop_mov = crop_mov * win

        # ------------------------------------------------------------------
        # Phase cross-correlation (shift detection)
        # ------------------------------------------------------------------
        #show_two_images(mask_ref, mask_mov)
        detected_shift, _, _ = phase_cross_correlation(
            crop_ref,
            crop_mov,
            reference_mask=mask_ref,
            moving_mask=mask_mov
        )

        print(f"Detected pixel offset (row, col): {-detected_shift}")

        # ------------------------------------------------------------------
        # Cost computation
        # ------------------------------------------------------------------
        ref_std = crop_ref[mask_ref].std()
        mov_std = crop_mov[mask_mov].std()
        avg_std = (ref_std + mov_std) / 2

        cost = 1.0 / (1e-6 + avg_std * mask_pixels_per * overlap_percentage)
        cost_list.append(cost)
        shift_list.append(detected_shift)

    return cost_list, shift_list

def show_two_images(img1, img2, title1="Image 1", title2="Image 2"):
    """
    Display two 2D images side by side using matplotlib with
    intensity range fixed to each image's min/max.

    Parameters
    ----------
    img1 : np.ndarray
        First image (2D).
    img2 : np.ndarray
        Second image (2D).
    title1 : str, optional
        Title for the first image.
    title2 : str, optional
        Title for the second image.
    """
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    vmin1, vmax1 = np.min(img1), np.max(img1)
    vmin2, vmax2 = np.min(img2), np.max(img2)

    axes[0].imshow(img1, cmap='gray', vmin=vmin1, vmax=vmax1)
    axes[0].set_title(title1)
    axes[0].axis('off')

    axes[1].imshow(img2, cmap='gray', vmin=vmin2, vmax=vmax2)
    axes[1].set_title(title2)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

def calculate_mask_roi(mask):
    """
    Compute the tight bounding box (ROI) around valid pixels in a boolean mask.

    Parameters
    ----------
    mask : np.ndarray (bool)
        Boolean array where True marks valid pixels and False invalid pixels.

    Returns
    -------
    x0 : int
        Left (minimum column index) of the ROI (inclusive).
    x1 : int
        Right (maximum column index) of the ROI (exclusive, suitable for slicing).
    y0 : int
        Top (minimum row index) of the ROI (inclusive).
    y1 : int
        Bottom (maximum row index) of the ROI (exclusive, suitable for slicing).

    Raises
    ------
    ValueError
        If the mask contains no valid (True) pixels.

    Notes
    -----
    - To crop an image `img` using this ROI, use:

        `img_cropped = img[y0:y1, x0:x1]`

      (row = y, col = x).
    """
    assert isinstance(mask, np.ndarray), "mask must be a numpy array"
    assert mask.dtype == bool, "mask must be a boolean array"

    xs, ys = np.nonzero(mask)
    if ys.size == 0:
        raise ValueError("Mask contains no valid (True) pixels.")

    y0, y1 = ys.min(), ys.max() + 1  # +1 to make it slice-exclusive
    x0, x1 = xs.min(), xs.max() + 1

    return x0, x1, y0, y1


In [8]:
from atlas.io import extract_s_number
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from collections import defaultdict, deque
import json

buffer_in_microns = 1

max_shift_in_pixles = 500

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file
            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=buffer_in_microns)
            
            first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
            extracted_number = extract_s_number(first_tif_path)
            # Define the output file path
            output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
            output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
            output_jason_path = raw_data_folder.parent.joinpath(f"transforms_S_{extracted_number}.json")

            if output_tif_path.exists():
                print(f"✅ Skipping: {output_tif_path.name} already exists.")
            else:
                print(f"🔄 Stitching image for S_{extracted_number}...")
                #stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=max_shift_in_pixles)

                try:
                    mif_tile_df = add_tile_overlap_columns(mif_tile_df)
                    # add info to the DF so we know where to find the images after they have been moved out of the scope
                    mif_tile_df['raw_data_folder'] = raw_data_folder
                    
                    # Calculate the costs of matching each tile to those that it overlaps with
                    n = len(mif_tile_df)
                    all_costs = []
                    all_shifts = []

                    for current_idx in range(n):
                        costs, shifts = match_tiles(mif_tile_df, reference_idx=current_idx, min_overlap_percent = 2, std_th=0, do_hanning=False)
                        # here I do the max displacement rule
                        for i, shift_i in enumerate(shifts):
                            d = np.linalg.norm(shift_i)
                            if d > max_shift_in_pixles:
                                costs[i] = 0.9
                                shifts[i] = np.array([0.0, 0.0])
                        
                        all_costs.append(costs)
                        all_shifts.append(shifts)

                    mif_tile_df["stitching_costs"] = all_costs
                    mif_tile_df["stitching_shifts"] = all_shifts


                    # Step 1: Build the cost matrix which I will use as adjacency for the min span tree
                    adj_matrix = build_adjacency_matrix_from_costs(mif_tile_df, cost_column='stitching_costs')

                    # Step 2: Create sparse matrix and compute MST
                    graph_sparse = csr_matrix(adj_matrix)
                    mst = minimum_spanning_tree(graph_sparse)
                    # The MST will be used to calculate the transofrmation matrices between each tile and a reference tile.
                    # For the moment I just pick 0 as reference but maybe there is a better way, in general I dont think it matters much.                
                    transform_dict = build_transform_dict_from_mst(mif_tile_df, mst, reference_tile=0)
                    
                    # apply transform, user inputs are transform_dict and mif_tile_df, output is the stitched_img
                    stitched_img = apply_transforms_and_stitch(mif_tile_df, transform_dict, reference_tile=0)
                    
                    # check if there is large dark areas around the obj
                    mask_valid = ~mask_low_and_saturation(stitched_img)
                    x0, x1, y0, y1 = calculate_mask_roi(mask_valid)
                    crop_img = stitched_img[x0:x1, y0:y1]

                    # Save the full image as a TIFF file
                    #tiff.imwrite(output_tif_path, np.flipud(stitched_img))
                    tiff.imwrite(output_tif_path, np.flipud(crop_img))
                    

                    mif_tile_df.to_csv(output_cc_path, index=False)

                    # Convert NumPy arrays to lists for JSON compatibility
                    json_ready_dict = {k: v.tolist() for k, v in transform_dict.items()}

                    # Save to JSON file
                    with open(output_jason_path, "w") as f:
                        json.dump(json_ready_dict, f, indent=2)
                        
                except Exception as e:
                    # Handle any error
                    print(f"Unexpected error with item {file}: {e}")
            
            


File with '.ve-mif' extension found: MosaicInfo_S_001_1502294429.ve-mif
🔄 Stitching image for S_1...

Processing reference tile 0...

Comparing reference 0 to tile 0...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 1...
Overlap %: 6
overlap img0 x: 7700-8192, y 0-8192
overlap img1 x: 0-492, y 0-8192
Detected pixel offset (row, col): [ 82. 145.]

Comparing reference 0 to tile 2...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 3...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 4...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 0 to tile 5...
Overlap %: 6
overlap img0 x: 0-8192, y 0-492
overlap img1 x: 0-8192, y 7700-8192
Detected pixel offset (row, col): [-39. -30.]

Comparing reference 0 to tile 6...
Overlap %: 0
Too little overlap: assigning cost=1.0 and shift=[0,0]

Comparing reference 